# Reto Week4 — Conversor de código con IA

---
**Autor:** José María Rivas | Práctica LLM Engineering
**Fecha:** 15 de julio de 2026
**Versión:** v3

---

## Resumen del proyecto

Herramienta Gradio que convierte código entre lenguajes de programación (Python ↔ C++)
usando modelos de IA, comparando el rendimiento y la corrección del resultado frente
al código original.

### Estructura — 3 pestañas

1. **Gestión de Prompts** — edición de los prompts de sistema/usuario que rigen la
   conversión, con restablecer a valores por defecto y enriquecimiento asistido por IA
   (modelos Frontera: OpenAI/Anthropic).
2. **Conversión de código** — selección de lenguaje origen/destino, gestión de archivos
   (cargar/guardar/eliminar), selector de modelo en cascada (filtro Frontera/Open-source
   → fabricante → modelo), conversión, ejecución independiente de cada lado, y
   comparación automática de resultados y tiempos.
3. **Historial de ejecuciones** — registro persistente (JSON Lines) de todas las
   conversiones probadas, con gestión para vaciarlo o recortarlo.

### Modelos soportados

- **Frontera:** OpenAI (GPT-4o), Anthropic (Claude Sonnet 5/Haiku 4.5), Google (Gemini 2.5 Flash)
- **Open-source (vía HuggingFace Inference Endpoints):** CodeQwen1.5-7B-Chat,
  Mistral-7B-Instruct-v0.3, Gemma-4-26B-A4B, Qwen2.5-Coder-32B-Instruct

### Notas técnicas

- Almacenamiento en disco local (código y log de benchmark), con capa de abstracción
  de E/S preparada para migrar a Google Drive sin tocar el resto del código.
- `SMOKE_TEST = False` — activar a `True` solo para verificaciones manuales puntuales;
  las pruebas gatean llamadas reales a las APIs para no consumir presupuesto sin querer.
---

In [1]:
# ============================================================
# SECCIÓN A — BLOQUE 1: Configuración inicial
# Imports, carga de credenciales, constantes de modelos y lenguajes
# ============================================================

In [2]:
# Librerías
import os
import re
import io
import json
import time
import subprocess
from datetime import datetime

from dotenv import load_dotenv
import pandas as pd
import gradio as gr

from openai import OpenAI
import anthropic
from google import genai
from google.genai import types
from huggingface_hub import InferenceClient
from transformers import AutoTokenizer


PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [3]:
# --- Interruptor global de pruebas: pon en True solo cuando quieras verificar manualmente ---
SMOKE_TEST = False

In [4]:
def texto_ayuda() -> str:
    return """
## Cómo usar esta herramienta

Esta app tiene tres pestañas, pensadas para usarse en este orden:

### 1. Gestión de Prompts
Aquí defines las instrucciones que se envían a la IA en cada conversión. Puedes editar
el prompt de sistema y el de usuario, restablecerlos a sus valores por defecto, o pedirle
a una IA (Frontera: OpenAI/Anthropic) que te proponga una versión mejorada a partir de
tus instrucciones. Los cambios aquí afectan a **todas** las conversiones posteriores,
sea cual sea el modelo elegido en la siguiente pestaña.

### 2. Conversión de código
- **Configuración de lenguajes:** elige el lenguaje de origen y destino.
- **Gestión de archivos:** carga código existente, guárdalo (origen, destino, o ambos —
  la extensión se añade sola según el lenguaje), o elimínalo.
- **Tipo de modelo:** filtra por Frontera/Open-source, elige fabricante y modelo.
- **Convertir:** solo traduce el código, no lo ejecuta.
- **Ejecutar Python / Ejecutar C++:** ejecuta cada lado por separado. Cuando ambos
  tienen resultado, se compara automáticamente (mismo resultado + tiempos) y se
  registra en el historial.

### 3. Historial de ejecuciones
Registro de todas las conversiones probadas: modelo usado, tiempos, si los resultados
coincidieron, y el speedup obtenido. Se puede vaciar por completo o recortar a las
últimas 10 ejecuciones con el botón "Gestionar historial".
"""

In [5]:
# --- Cargar variables de entorno ---
load_dotenv(override=True)

API_KEYS = {
    "openai":    os.getenv("OPENAI_API_KEY"),
    "anthropic": os.getenv("ANTHROPIC_API_KEY"),
    "groq":      os.getenv("GROQ_API_KEY"),
    "deepseek":  os.getenv("DEEPSEEK_API_KEY"),
    "gemini":    os.getenv("GOOGLE_API_KEY"),
    "huggingface": os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_TOKEN")
}


In [6]:
# Smoke test rápido — comprobar que las claves se han cargado antes de seguir
if SMOKE_TEST:
    for proveedor, clave in API_KEYS.items():
        estado = f"{clave[:6]}...{clave[-4:]}" if clave else "NO ENCONTRADA"
        print(f"{proveedor}: {estado}")

openai: sk-pro...4DYA
anthropic: sk-ant...agAA
groq: gsk_zk...EMaY
deepseek: NO ENCONTRADA
gemini: AIzaSy...vSmw
huggingface: hf_UQy...wNuy


In [7]:
# --- Clientes de las IAs Frontera (patrón client_nombreIA) ---
client_openai = OpenAI(api_key=API_KEYS["openai"])
client_claude = anthropic.Anthropic(api_key=API_KEYS["anthropic"])
client_genai = genai.Client(api_key=API_KEYS["gemini"])

client_hf = InferenceClient(token=API_KEYS["huggingface"])  # reutilizable: el modelo se indica en cada llamada

In [8]:
# ============================================================
# Lenguajes soportados (ampliable sin tocar el resto del código)
# ============================================================
LANGUAGES = ["Python", "C++"]
LENGUAJE_ORIGEN_DEFAULT = "Python"
LENGUAJE_DESTINO_DEFAULT = "C++"

In [10]:
# ============================================================
# Modelos disponibles para la CONVERSIÓN de código
# Los open-source solo se añaden si su URL de endpoint existe en .env
# ============================================================
MODELOS = {
    "OpenAI":    {"tipo": "Frontera", "modelos": {"gpt-4o": "gpt-4o", "gpt-4o-mini": "gpt-4o-mini"}},
    "Anthropic": {"tipo": "Frontera", "modelos": {"claude-sonnet-5": "claude-sonnet-5", "claude-haiku-4.5": "claude-haiku-4-5-20251001"}},
    "Google": {"tipo": "Frontera", "modelos": {"gemini-2.5-flash": "gemini-2.5-flash", "gemini-2.5-flash-lite": "gemini-2.5-flash-lite"}},
}

if os.environ.get("CODE_QWEN_URL"):
    MODELOS["CodeQwen"] = {"tipo": "Open-source", "modelos": {"CodeQwen1.5-7B-Chat": os.environ["CODE_QWEN_URL"]}}
if os.environ.get("MISTRAL_URL"):
    MODELOS["Mistral"] = {"tipo": "Open-source", "modelos": {"Mistral-7B-Instruct-v0.3": os.environ["MISTRAL_URL"]}}
if os.environ.get("GEMMA4_URL"):
    MODELOS["Gemma4"] = {"tipo": "Open-source", "modelos": {"Gemma-4-26B-A4B": os.environ["GEMMA4_URL"]}}
if os.environ.get("QWEN_CODER_URL"):
    MODELOS["Qwen2.5-Coder"] = {"tipo": "Open-source", "modelos": {"Qwen2.5-Coder-32B-Instruct": os.environ["QWEN_CODER_URL"]}}

In [11]:
# --- Mapeo URL de endpoint → repo de HuggingFace del que sacar el tokenizer ---

TOKENIZER_REPO_POR_URL = {}
if os.environ.get("CODE_QWEN_URL"):
    TOKENIZER_REPO_POR_URL[os.environ["CODE_QWEN_URL"]] = "Qwen/CodeQwen1.5-7B-Chat"
if os.environ.get("MISTRAL_URL"):
    TOKENIZER_REPO_POR_URL[os.environ["MISTRAL_URL"]] = "mistralai/Mistral-7B-Instruct-v0.3"
if os.environ.get("QWEN_CODER_URL"):
    TOKENIZER_REPO_POR_URL[os.environ["QWEN_CODER_URL"]] = "Qwen/Qwen2.5-Coder-32B-Instruct"

_tokenizers_cache = {}

def _construir_prompt_con_template(mensajes, modelo_url):
    """Si conocemos el tokenizer del modelo, aplica su chat template real (ChatML, etc.)
    en vez de concatenar el prompt en plano — necesario para que un modelo instruct
    entienda que debe seguir las instrucciones, no solo continuar el texto."""
    repo_id = TOKENIZER_REPO_POR_URL.get(modelo_url)
    if not repo_id:
        return f"{mensajes[0]['content']}\n\n{mensajes[1]['content']}"

    if repo_id not in _tokenizers_cache:
        try:
            _tokenizers_cache[repo_id] = AutoTokenizer.from_pretrained(repo_id)
        except Exception:
            _tokenizers_cache[repo_id] = None

    tokenizer = _tokenizers_cache[repo_id]
    if tokenizer is None:
        return f"{mensajes[0]['content']}\n\n{mensajes[1]['content']}"

    return tokenizer.apply_chat_template(mensajes, tokenize=False, add_generation_prompt=True)


def _fallback_text_generation(mensajes, modelo_url):
    prompt = _construir_prompt_con_template(mensajes, modelo_url)
    texto = client_hf.text_generation(prompt, model=modelo_url, max_new_tokens=2000, return_full_text=False)
    if texto.startswith(prompt):
        texto = texto[len(prompt):]
    yield limpiar_markdown(texto)

In [12]:
if SMOKE_TEST:
    mensajes_prueba = [
        {"role": "system", "content": "Eres un asistente útil."},
        {"role": "user", "content": "Responde solo con OK"}
    ]
    prompt_construido = _construir_prompt_con_template(mensajes_prueba, os.environ["CODE_QWEN_URL"])
    print("--- Prompt construido ---")
    print(prompt_construido)
    texto = client_hf.text_generation(prompt_construido, model=os.environ["CODE_QWEN_URL"], max_new_tokens=20)
    print("--- Respuesta ---")
    print(texto)

--- Prompt construido ---
<|im_start|>system
Eres un asistente útil.<|im_end|>
<|im_start|>user
Responde solo con OK<|im_end|>
<|im_start|>assistant



HfHubHTTPError: Server error '500 Internal Server Error' for url 'https://okip7vd18g237v8o.us-east4.gcp.endpoints.huggingface.cloud' (Request ID: pfIohZ)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/500

In [13]:
# --- Modelos Frontera exclusivos para el Tab 2 (enriquecer prompts) ---
MODELOS_ENRIQUECIMIENTO = {
    "OpenAI": MODELOS["OpenAI"]["modelos"],
    "Anthropic": MODELOS["Anthropic"]["modelos"],
}

In [14]:
# --- Almacena modelos frontera actualizados para el Tab 1 (Bloque B - Gestión de Prompts) ---
_cache_modelos_ia = {}

In [15]:
# ============================================================
# Prompts por defecto (con placeholders de lenguaje)
# ============================================================
DEFAULT_SYSTEM_PROMPT = (
    "Eres un asistente experto en programación que convierte código {lenguaje_origen} "
    "en código {lenguaje_destino} equivalente y de alto rendimiento. "
    "Responde ÚNICAMENTE con el código {lenguaje_destino}, sin explicaciones, "
    "sin bloques de markdown (```), y sin ninguna palabra antes o después del código."
)

DEFAULT_USER_PROMPT_TEMPLATE = (
    "Reescribe el siguiente código {lenguaje_origen} en {lenguaje_destino}, "
    "con la implementación más rápida posible. "
    "El código {lenguaje_destino} debe producir exactamente el mismo resultado:\n\n{codigo}"
)


In [16]:
# ============================================================
# Log de ejecuciones (benchmark) — disco local por ahora, JSON Lines
# ============================================================
if SMOKE_TEST:
    BENCHMARK_LOG_PATH = "benchmark_log.jsonl"
    
    print("\nBloque 1 completado.")
    print("LANGUAGES:", LANGUAGES)
    print("Fabricantes de modelos cargados:", list(MODELOS.keys()))


Bloque 1 completado.
LANGUAGES: ['Python', 'C++']
Fabricantes de modelos cargados: ['OpenAI', 'Anthropic', 'Google', 'CodeQwen', 'Mistral', 'Gemma4', 'Qwen2.5-Coder']


In [17]:
# ============================================================
# Estructura de carpetas locales (mismo patrón que Week3 con Drive:
# rutas explícitas + creación automática si no existen)
# ============================================================
BASE_DIR = os.getcwd()
print("Directorio de trabajo actual:", BASE_DIR)

CARPETA_ORIGEN  = os.path.join(BASE_DIR, "codigo_origen")
CARPETA_DESTINO = os.path.join(BASE_DIR, "codigo_destino")
CARPETA_LOGS    = os.path.join(BASE_DIR, "logs")

for carpeta in [CARPETA_ORIGEN, CARPETA_DESTINO, CARPETA_LOGS]:
    os.makedirs(carpeta, exist_ok=True)
    print(f"Carpeta lista: {carpeta}")

# --- Ruta del log de benchmark, ya dentro de su carpeta dedicada ---
BENCHMARK_LOG_PATH = os.path.join(CARPETA_LOGS, "benchmark_log.jsonl")

Directorio de trabajo actual: c:\Users\Jose M. Rivas\Developer\Github\llm_engineering\week4
Carpeta lista: c:\Users\Jose M. Rivas\Developer\Github\llm_engineering\week4\codigo_origen
Carpeta lista: c:\Users\Jose M. Rivas\Developer\Github\llm_engineering\week4\codigo_destino
Carpeta lista: c:\Users\Jose M. Rivas\Developer\Github\llm_engineering\week4\logs


In [18]:
# ============================================================
# SECCIÓN A — BLOQUE 2: Capa de abstracción de E/S + listado de archivos
# Implementación LOCAL (misma firma que tendrá la versión Drive más adelante)
# ============================================================

In [19]:
# --- Extensión de archivo según el lenguaje (ampliable junto con LANGUAGES) ---
EXTENSIONES = {
    "Python": ".py",
    "C++": ".cpp",
}

In [20]:
def leer_archivo(nombre: str, carpeta: str) -> str:
    """Lee el contenido de un archivo de texto dentro de una carpeta dada."""
    ruta = os.path.join(carpeta, nombre)
    with open(ruta, "r", encoding="utf-8") as f:
        return f.read()

In [21]:
def escribir_archivo(nombre: str, contenido: str, carpeta: str) -> str:
    """Escribe (o sobrescribe) un archivo de texto dentro de una carpeta dada. Devuelve la ruta completa."""
    ruta = os.path.join(carpeta, nombre)
    with open(ruta, "w", encoding="utf-8") as f:
        f.write(contenido)
    return ruta

In [22]:
def listar_archivos(carpeta: str, extension: str = None) -> list[str]:
    """Lista los archivos de una carpeta, opcionalmente filtrados por extensión."""
    if not os.path.exists(carpeta):
        return []
    archivos = [a for a in os.listdir(carpeta) if os.path.isfile(os.path.join(carpeta, a))]
    if extension:
        archivos = [a for a in archivos if a.endswith(extension)]
    return sorted(archivos)

In [23]:
def eliminar_archivo(nombre: str, carpeta: str) -> bool:
    """Elimina un único archivo de una carpeta dada. Devuelve True si se eliminó, False si no existía."""
    ruta = os.path.join(carpeta, nombre)
    if os.path.exists(ruta):
        os.remove(ruta)
        return True
    return False


In [24]:
def append_jsonl(registro: dict, log_path: str = BENCHMARK_LOG_PATH):
    """Añade una línea (un registro JSON) al log de ejecuciones, sin reescribir el archivo completo."""
    with open(log_path, "a", encoding="utf-8") as f:
        f.write(json.dumps(registro, ensure_ascii=False) + "\n")

In [25]:
# ============================================================
# Listado inicial de archivos, para poblar los desplegables del Tab 1
# ============================================================
def refrescar_listado_archivos(lenguaje: str, carpeta: str) -> list[str]:
    """Devuelve los archivos disponibles en una carpeta, filtrados por la extensión del lenguaje."""
    extension = EXTENSIONES.get(lenguaje)
    return listar_archivos(carpeta, extension)

archivos_origen_iniciales = refrescar_listado_archivos(LENGUAJE_ORIGEN_DEFAULT, CARPETA_ORIGEN)
archivos_destino_iniciales = refrescar_listado_archivos(LENGUAJE_DESTINO_DEFAULT, CARPETA_DESTINO)

if SMOKE_TEST:
    print("Archivos en codigo_origen:", archivos_origen_iniciales or "(carpeta vacía)")
    print("Archivos en codigo_destino:", archivos_destino_iniciales or "(carpeta vacía)")

Archivos en codigo_origen: ['Prueba_1.py', 'convertido.py', 'prueba_pi.py', 'temp_origen.py']
Archivos en codigo_destino: ['AD_CodeQwen_157BChat.cpp', 'Prueba_1.cpp', 'Prueba_2.cpp', 'convertido.cpp', 'prueba_pi.cpp', 'temp_destino.cpp']


In [26]:
# ============================================================
# SECCIÓN A — BLOQUE 3: Construcción dinámica de prompts
# Genera el prompt de sistema y de usuario final según lenguaje
# origen/destino, usando los prompts "activos" (editables desde el Tab 2)
# ============================================================

In [27]:
# --- Prompts activos: variables globales inicializadas con los valores por defecto del Bloque 1 ---
# (Cuando construyamos el Tab 2, sus botones de "Restablecer"/"Aceptar propuesta"
#  modificarán estas mismas variables con `global`, en vez de duplicar el estado)
prompt_sistema_activo = DEFAULT_SYSTEM_PROMPT
prompt_usuario_activo = DEFAULT_USER_PROMPT_TEMPLATE

In [28]:
def construir_prompt_sistema(lenguaje_origen: str, lenguaje_destino: str) -> str:
    """Sustituye los placeholders del prompt de sistema activo por los lenguajes seleccionados."""
    return prompt_sistema_activo.format(
        lenguaje_origen=lenguaje_origen,
        lenguaje_destino=lenguaje_destino
    )

In [29]:
def construir_prompt_usuario(lenguaje_origen: str, lenguaje_destino: str, codigo: str,
                               instrucciones_adicionales: str = None) -> str:
    """Sustituye los placeholders del prompt de usuario activo, inserta el código
    y añade instrucciones adicionales puntuales si el usuario las escribió en el Tab 1."""
    prompt = prompt_usuario_activo.format(
        lenguaje_origen=lenguaje_origen,
        lenguaje_destino=lenguaje_destino,
        codigo=codigo
    )
    if instrucciones_adicionales:
        prompt += f"\n\nInstrucciones adicionales para esta conversión: {instrucciones_adicionales}"
    return prompt


In [30]:
def messages_for(lenguaje_origen: str, lenguaje_destino: str, codigo: str,
                  instrucciones_adicionales: str = None) -> list[dict]:
    """Ensambla la lista de mensajes (system + user) lista para enviar a cualquier modelo Frontera.
    Los modelos Open-source la adaptan a su propio formato de chat template en el Bloque 5."""
    return [
        {"role": "system", "content": construir_prompt_sistema(lenguaje_origen, lenguaje_destino)},
        {"role": "user", "content": construir_prompt_usuario(lenguaje_origen, lenguaje_destino, codigo, instrucciones_adicionales)},
    ]



In [31]:
# --- Smoke test rápido con un fragmento de código de prueba ---

if SMOKE_TEST:    
    codigo_prueba = "print('hola mundo')"
    mensajes_prueba = messages_for("Python", "C++", codigo_prueba)
    
    print("--- Prompt de sistema ---")
    print(mensajes_prueba[0]["content"])
    print("\n--- Prompt de usuario ---")
    print(mensajes_prueba[1]["content"])

--- Prompt de sistema ---
Eres un asistente experto en programación que convierte código Python en código C++ equivalente y de alto rendimiento. Responde ÚNICAMENTE con el código C++, sin explicaciones, sin bloques de markdown (```), y sin ninguna palabra antes o después del código.

--- Prompt de usuario ---
Reescribe el siguiente código Python en C++, con la implementación más rápida posible. El código C++ debe producir exactamente el mismo resultado:

print('hola mundo')


In [32]:
# ===============================================================================
# SECCIÓN B — BLOQUE 1: Funciones específicas para el área de gestión de Prompts
# ===============================================================================

In [33]:
# --- Restablecer prompt de sistema ---
def restablecer_prompt_sistema():
    global prompt_sistema_activo
    prompt_sistema_activo = DEFAULT_SYSTEM_PROMPT
    return gr.update(value=DEFAULT_SYSTEM_PROMPT), "Prompt de sistema restablecido a valores por defecto."

In [34]:
# --- Restablecer prompt de usuario ---
def restablecer_prompt_usuario():
    global prompt_usuario_activo
    prompt_usuario_activo = DEFAULT_USER_PROMPT_TEMPLATE
    return gr.update(value=DEFAULT_USER_PROMPT_TEMPLATE), "Prompt de usuario restablecido a valores por defecto."

In [35]:
# --- Guardar prompt de sistema como activo ---
def guardar_prompt_sistema(valor):
    global prompt_sistema_activo
    if not valor or not valor.strip():
        return "El prompt de sistema no puede quedar vacío. No se ha guardado."
    prompt_sistema_activo = valor
    return "Guardado como prompt de sistema activo."

In [36]:
# --- Guardar prompt de usuario como activo ---
def guardar_prompt_usuario(valor):
    global prompt_usuario_activo
    if not valor or not valor.strip():
        return "El prompt de usuario no puede quedar vacío. No se ha guardado."
    prompt_usuario_activo = valor
    return "Guardado como prompt de usuario activo."

In [37]:
#--- Recargar prompt de sistema activo (descarta ediciones sin guardar) ---
def recargar_prompt_sistema():
    return gr.update(value=prompt_sistema_activo), "Recargado el prompt de sistema activo (se han descartado cambios sin guardar)."

In [38]:
#--- Recargar prompt de usuario activo (descarta ediciones sin guardar) ---
def recargar_prompt_usuario():
    return gr.update(value=prompt_usuario_activo), "Recargado el prompt de usuario activo (se han descartado cambios sin guardar)."

In [39]:
#---  Comprueba caché sino actualiza los modelos Frontera del fabricante --- 
def actualizar_modelos_enriquecimiento(fabricante):
    if fabricante in _cache_modelos_ia:
        modelos = _cache_modelos_ia[fabricante]
    else:
        modelos = list(MODELOS_ENRIQUECIMIENTO.get(fabricante, {}).keys())
    valor = modelos[0] if modelos else None
    return gr.update(choices=modelos, value=valor)

In [40]:
#--- Actualizar modelos consultando la API en vivo ---
def actualizar_modelos_ia_api(fabricante):
    try:
        if fabricante == "OpenAI":
            respuesta = client_openai.models.list()
            nombres = sorted([m.id for m in respuesta.data if "gpt" in m.id])
        elif fabricante == "Anthropic":
            respuesta = client_claude.models.list()
            nombres = sorted([m.id for m in respuesta.data])
        else:
            nombres = []

        if nombres:
            _cache_modelos_ia[fabricante] = nombres
        else:
            nombres = list(MODELOS_ENRIQUECIMIENTO.get(fabricante, {}).keys())

        valor = nombres[0] if nombres else None
        return gr.update(choices=nombres, value=valor), f"Modelos actualizados desde la API ({len(nombres)} encontrados)."
    except Exception as e:
        modelos_fallback = _cache_modelos_ia.get(fabricante) or list(MODELOS_ENRIQUECIMIENTO.get(fabricante, {}).keys())
        valor_fallback = modelos_fallback[0] if modelos_fallback else None
        return gr.update(choices=modelos_fallback, value=valor_fallback), f"No se pudo consultar la API ({e}); usando lista local."

In [41]:
#--- Enriquecer un prompt con IA ---
def enriquecer_prompt(cual, prompt_sistema_actual, prompt_usuario_actual, instrucciones, fabricante, modelo_mostrado):
    if not instrucciones or not instrucciones.strip():
        return "Escribe primero qué quieres cambiar o mejorar del prompt.", gr.update()

    prompt_actual = prompt_sistema_actual if cual == "Prompt de sistema" else prompt_usuario_actual
    modelo_real = resolver_modelo(fabricante, modelo_mostrado)

    sistema_meta = (
        "Eres un experto en prompt engineering. Se te va a dar un prompt existente y unas "
        "instrucciones de cómo mejorarlo o modificarlo. Devuelve ÚNICAMENTE el nuevo prompt "
        "reescrito, sin explicaciones, sin comillas, sin texto adicional antes o después."
    )
    usuario_meta = (
        f"Prompt actual ({cual}):\n{prompt_actual}\n\n"
        f"Instrucciones de mejora: {instrucciones}\n\n"
        f"Devuelve el prompt reescrito completo, manteniendo los placeholders "
        f"{{lenguaje_origen}}, {{lenguaje_destino}} y {{codigo}} si existen en el original."
    )

    try:
        if fabricante == "OpenAI":
            respuesta = client_openai.chat.completions.create(
                model=modelo_real,
                messages=[{"role": "system", "content": sistema_meta}, {"role": "user", "content": usuario_meta}],
            )
            propuesta = respuesta.choices[0].message.content
        elif fabricante == "Anthropic":
            respuesta = client_claude.messages.create(
                model=modelo_real, max_tokens=1000,
                system=sistema_meta, messages=[{"role": "user", "content": usuario_meta}],
            )
            propuesta = respuesta.content[0].text
        else:
            return f"Fabricante no soportado para enriquecimiento: {fabricante}", gr.update()

        return "Propuesta generada. Revísala y pulsa 'Aceptar propuesta' si te vale.", gr.update(value=propuesta.strip())
    except Exception as e:
        return f"Error al generar la propuesta: {e}", gr.update()

In [42]:
#--- Aceptar la propuesta como nuevo prompt activo ---
def aceptar_propuesta(cual, propuesta):
    global prompt_sistema_activo, prompt_usuario_activo
    if not propuesta or not propuesta.strip():
        return gr.update(), gr.update(), "No hay ninguna propuesta que aceptar."

    if cual == "Prompt de sistema":
        prompt_sistema_activo = propuesta
        return gr.update(value=propuesta), gr.update(), "Propuesta aplicada como prompt de sistema activo."
    else:
        prompt_usuario_activo = propuesta
        return gr.update(), gr.update(value=propuesta), "Propuesta aplicada como prompt de usuario activo."

In [43]:
#--- Descartar la propuesta ---
def descartar_propuesta():
    return ""

In [44]:
# ============================================================
# SECCIÓN A — BLOQUE 4: Funciones de streaming por modelo
# ============================================================

In [45]:
def limpiar_markdown(texto: str) -> str:
    """Quita bloques de markdown que algunos modelos añaden pese a las instrucciones."""
    texto = re.sub(r'^```[a-zA-Z]*\n?', '', texto.strip())
    texto = re.sub(r'\n?```$', '', texto.strip())
    return texto.strip()

In [46]:
def reforzar_prompt_open_source(mensajes: list[dict]) -> list[dict]:
    """Añade una instrucción extra al último mensaje de usuario. Se aplica a
    CUALQUIER modelo de tipo Open-source, no hardcodeado por nombre."""
    refuerzo = (
        "\n\nIMPORTANTE: tu respuesta debe empezar directamente con el código. "
        "No uses bloques de markdown (```), no escribas ninguna palabra de explicación, "
        "ni antes ni después del código. Solo código, listo para compilar/ejecutar."
    )
    mensajes = [m.copy() for m in mensajes]
    mensajes[-1]["content"] += refuerzo
    return mensajes

In [47]:
# --- Frontera ---

def stream_gpt(lenguaje_origen, lenguaje_destino, codigo, modelo, instrucciones_adicionales=None):
    mensajes = messages_for(lenguaje_origen, lenguaje_destino, codigo, instrucciones_adicionales)
    stream = client_openai.chat.completions.create(model=modelo, messages=mensajes, stream=True)
    reply = ""
    for chunk in stream:
        reply += chunk.choices[0].delta.content or ""
        yield limpiar_markdown(reply)

In [48]:
def stream_claude(lenguaje_origen, lenguaje_destino, codigo, modelo, instrucciones_adicionales=None):
    mensajes = messages_for(lenguaje_origen, lenguaje_destino, codigo, instrucciones_adicionales)
    reply = ""
    with client_claude.messages.stream(
        model=modelo,
        max_tokens=2000,
        system=mensajes[0]["content"],
        messages=[{"role": "user", "content": mensajes[1]["content"]}]
    ) as stream:
        for texto in stream.text_stream:
            reply += texto
            yield limpiar_markdown(reply)

In [49]:
def stream_gemini(lenguaje_origen, lenguaje_destino, codigo, modelo, instrucciones_adicionales=None):
    mensajes = messages_for(lenguaje_origen, lenguaje_destino, codigo, instrucciones_adicionales)
    reply = ""
    stream = client_genai.models.generate_content_stream(
        model=modelo,
        contents=mensajes[1]["content"],
        config=types.GenerateContentConfig(system_instruction=mensajes[0]["content"]),
    )
    for chunk in stream:
        if chunk.text:
            reply += chunk.text
            yield limpiar_markdown(reply)


In [50]:
# --- Open-source (genérica para cualquier modelo del dict MODELOS con tipo Open-source) ---

def stream_open_source(lenguaje_origen, lenguaje_destino, codigo, modelo_url, instrucciones_adicionales=None,
                        max_reintentos=5, espera_segundos=15):
    mensajes = messages_for(lenguaje_origen, lenguaje_destino, codigo, instrucciones_adicionales)
    mensajes = reforzar_prompt_open_source(mensajes)

    for intento in range(1, max_reintentos + 1):
        try:
            respuesta = client_hf.chat_completion(messages=mensajes, model=modelo_url, max_tokens=2000)
            texto = respuesta.choices[0].message.content
            yield limpiar_markdown(texto)
            return
        except Exception as e:
            error_str = str(e)
            if "503" in error_str and intento < max_reintentos:
                yield f"⏳ El endpoint está despertando (intento {intento}/{max_reintentos}), esperando {espera_segundos}s..."
                time.sleep(espera_segundos)
                continue
            if "404" in error_str:
                yield from _fallback_text_generation(mensajes, modelo_url)
                return
            raise

In [51]:
# ============================================================
# Orquestadora: elige la función según el fabricante
# ============================================================
def convertir_codigo(fabricante, modelo, lenguaje_origen, lenguaje_destino, codigo, instrucciones_adicionales=None):
    tipo = MODELOS[fabricante]["tipo"]

    if fabricante == "OpenAI":
        yield from stream_gpt(lenguaje_origen, lenguaje_destino, codigo, modelo, instrucciones_adicionales)
    elif fabricante == "Anthropic":
        yield from stream_claude(lenguaje_origen, lenguaje_destino, codigo, modelo, instrucciones_adicionales)
    elif fabricante == "Google":
        yield from stream_gemini(lenguaje_origen, lenguaje_destino, codigo, modelo, instrucciones_adicionales)
    elif tipo == "Open-source":
        yield from stream_open_source(lenguaje_origen, lenguaje_destino, codigo, modelo, instrucciones_adicionales)
    else:
        raise ValueError(f"Fabricante no reconocido: {fabricante}")

In [52]:
# --- Smoke test con GPT-4o-mini (rápido y barato, para validar el flujo sin gastar de más) ---
if SMOKE_TEST:
    for parcial in convertir_codigo("OpenAI", "gpt-4o-mini", "Python", "C++", "print('hola mundo')"):
        resultado_prueba = parcial
    print("\n--- Resultado ---")
    print(resultado_prueba)


--- Resultado ---
#include <iostream>

int main() {
    std::cout << "hola mundo" << std::endl;
    return 0;
}


In [53]:
# ============================================================
# SECCIÓN A — BLOQUE 5: Ejecución de código (genérica por lenguaje)
# ============================================================

In [54]:
# --- Cómo ejecutar cada lenguaje, siguiendo el mismo patrón que MODELOS/EXTENSIONES ---

EJECUTORES = {
    "Python": {
        "tipo": "interpretado",
        "comando": lambda archivo: ["python", archivo],
    },
    "C++": {
        "tipo": "compilado",
        "comando_compilar": lambda archivo, exe: ["g++", "-O3", "-std=c++17", "-march=native", "-o", exe, archivo],
        "comando_ejecutar": lambda exe: [exe],
    },
}

In [55]:
def execute_code(lenguaje: str, codigo: str, nombre_base: str, carpeta: str) -> dict:
    """Escribe el código a disco, lo ejecuta (compilando antes si hace falta) y
    devuelve stdout, stderr, tiempo de ejecución (medido externamente) y returncode."""

    if lenguaje not in EJECUTORES:
        raise ValueError(f"No hay ejecutor definido para el lenguaje: {lenguaje}")

    ejecutor = EJECUTORES[lenguaje]
    extension = EXTENSIONES[lenguaje]
    nombre_archivo = f"{nombre_base}{extension}"
    ruta_archivo = escribir_archivo(nombre_archivo, codigo, carpeta)

    try:
        if ejecutor["tipo"] == "interpretado":
            comando = ejecutor["comando"](ruta_archivo)
            inicio = time.time()
            resultado = subprocess.run(comando, capture_output=True, text=True, check=True)
            tiempo = time.time() - inicio

        elif ejecutor["tipo"] == "compilado":
            ruta_exe = os.path.join(carpeta, f"{nombre_base}.exe")
            compile_cmd = ejecutor["comando_compilar"](ruta_archivo, ruta_exe)
            subprocess.run(compile_cmd, capture_output=True, text=True, check=True)

            inicio = time.time()
            resultado = subprocess.run(ejecutor["comando_ejecutar"](ruta_exe), capture_output=True, text=True, check=True)
            tiempo = time.time() - inicio

        return {
            "stdout": resultado.stdout.strip(),
            "stderr": resultado.stderr.strip(),
            "tiempo": round(tiempo, 6),
            "error": None,
        }

    except subprocess.CalledProcessError as e:
        return {"stdout": None, "stderr": e.stderr, "tiempo": None, "error": str(e)}


In [56]:
# --- Smoke test: mismo ejemplo de pi usado durante todo el Día 3/4 ---

if SMOKE_TEST:
    codigo_python_prueba = """import time
    
    def calculate(iterations, param1, param2):
        result = 1.0
        for i in range(1, iterations+1):
            j = i * param1 - param2
            result -= (1/j)
            j = i * param1 + param2
            result += (1/j)
        return result
    
    result = calculate(10_000_000, 4, 1) * 4
    print(f"{result:.12f}")
    """
    
    resultado_python = execute_code("Python", codigo_python_prueba, "prueba_pi", CARPETA_ORIGEN) 
    print("Python:", resultado_python)

Python: {'stdout': None, 'stderr': '  File "c:\\Users\\Jose M. Rivas\\Developer\\Github\\llm_engineering\\week4\\codigo_origen\\prueba_pi.py", line 3\n    def calculate(iterations, param1, param2):\nIndentationError: unexpected indent\n', 'tiempo': None, 'error': "Command '['python', 'c:\\\\Users\\\\Jose M. Rivas\\\\Developer\\\\Github\\\\llm_engineering\\\\week4\\\\codigo_origen\\\\prueba_pi.py']' returned non-zero exit status 1."}


In [57]:
# ============================================================
# SECCIÓN A — BLOQUE 6: Benchmark automatizado + registro en JSONL
# ============================================================

In [58]:
def cargar_historial_benchmark() -> pd.DataFrame:
    """Recarga el historial completo desde el JSONL — única fuente de verdad,
    sobrevive a reinicios de kernel."""
    columnas = ["nº", "timestamp", "archivo_origen", "archivo_destino", "lenguaje_origen", "lenguaje_destino",
                "fabricante", "modelo", "tipo_modelo", "tiempo_origen_s", "tiempo_destino_s",
                "resultado_origen", "resultado_destino", "resultados_coinciden", "speedup"]
    if not os.path.exists(BENCHMARK_LOG_PATH) or os.path.getsize(BENCHMARK_LOG_PATH) == 0:
        return pd.DataFrame(columns=columnas)
    df = pd.read_json(BENCHMARK_LOG_PATH, lines=True)
    df["timestamp"] = df["timestamp"].astype(str)
    for col in ["resultado_origen", "resultado_destino"]:
        df[col] = df[col].astype(str).str.replace("\n", " | ", regex=False)
    df.insert(0, "nº", range(1, len(df) + 1))
    return df

In [59]:
def gestionar_historial(accion):
    if not os.path.exists(BENCHMARK_LOG_PATH):
        df = cargar_historial_benchmark()
        return "No hay historial que gestionar.", df, contar_registros(df)

    if accion == "Vaciar todo":
        open(BENCHMARK_LOG_PATH, "w", encoding="utf-8").close()
        mensaje = "Historial vaciado por completo."
    elif accion == "Mantener últimos 10":
        with open(BENCHMARK_LOG_PATH, "r", encoding="utf-8") as f:
            lineas = f.readlines()
        ultimas = lineas[-10:]
        with open(BENCHMARK_LOG_PATH, "w", encoding="utf-8") as f:
            f.writelines(ultimas)
        mensaje = f"Historial recortado a las últimas {len(ultimas)} ejecuciones."
    else:
        mensaje = "Selecciona primero qué acción quieres hacer."

    df = cargar_historial_benchmark()
    return mensaje, df, contar_registros(df)

In [60]:
def ejecutar_benchmark(fabricante: str, modelo: str, lenguaje_origen: str, lenguaje_destino: str,
                        codigo_origen: str, nombre_archivo_origen: str, nombre_archivo_destino: str,
                        instrucciones_adicionales: str = None) -> dict:
    """Convierte el código, ejecuta origen y destino, compara resultados,
    registra la ejecución en el log JSONL, y devuelve el resumen para la interfaz."""

    timestamp = datetime.now().isoformat(timespec="seconds")

    # --- 1. Conversión ---
    codigo_destino = ""
    for parcial in convertir_codigo(fabricante, modelo, lenguaje_origen, lenguaje_destino,
                                     codigo_origen, instrucciones_adicionales):
        codigo_destino = parcial

    nombre_base_origen = os.path.splitext(nombre_archivo_origen)[0]
    nombre_base_destino = os.path.splitext(nombre_archivo_destino)[0]

    # --- 2. Ejecución de origen y destino ---
    resultado_origen = execute_code(lenguaje_origen, codigo_origen, nombre_base_origen, CARPETA_ORIGEN)
    resultado_destino = execute_code(lenguaje_destino, codigo_destino, nombre_base_destino, CARPETA_DESTINO)

    # --- 3. Comparación ---
    coinciden = (
        resultado_origen["error"] is None
        and resultado_destino["error"] is None
        and resultado_origen["stdout"] == resultado_destino["stdout"]
    )
    speedup = None
    if resultado_origen["tiempo"] and resultado_destino["tiempo"]:
        speedup = round(resultado_origen["tiempo"] / resultado_destino["tiempo"], 2)

    # --- 4. Registro en el log (JSON Lines, append seguro) ---
    registro = {
        "timestamp": timestamp,
        "archivo_origen": nombre_archivo_origen,
        "archivo_destino": nombre_archivo_destino,
        "lenguaje_origen": lenguaje_origen,
        "lenguaje_destino": lenguaje_destino,
        "fabricante": fabricante,
        "modelo": modelo,
        "tipo_modelo": MODELOS[fabricante]["tipo"],
        "tiempo_origen_s": resultado_origen["tiempo"],
        "tiempo_destino_s": resultado_destino["tiempo"],
        "resultado_origen": resultado_origen["stdout"],
        "resultado_destino": resultado_destino["stdout"],
        "resultados_coinciden": coinciden,
        "speedup": speedup,
    }
    append_jsonl(registro)

    # --- 5. Guardar el código destino generado (para que quede disponible en CARPETA_DESTINO) ---
    escribir_archivo(nombre_archivo_destino, codigo_destino, CARPETA_DESTINO)

    return {
        "codigo_destino": codigo_destino,
        "resultado_origen": resultado_origen,
        "resultado_destino": resultado_destino,
        "coinciden": coinciden,
        "speedup": speedup,
        "registro": registro,
    }


In [61]:
def contar_registros(df: pd.DataFrame) -> str:
    return f"**Total de registros: {len(df)}**"

In [62]:
def cargar_historial_al_entrar():
    df = cargar_historial_benchmark()
    return df, contar_registros(df)

In [63]:
# --- Smoke test: benchmark completo con el mismo ejemplo de pi ---

if SMOKE_TEST:
    resumen = ejecutar_benchmark(
        fabricante="OpenAI",
        modelo="gpt-4o-mini",
        lenguaje_origen="Python",
        lenguaje_destino="C++",
        codigo_origen=codigo_python_prueba,
        nombre_archivo_origen="prueba_pi.py",
        nombre_archivo_destino="prueba_pi.cpp",
    )
    
    print(f"Coinciden: {resumen['coinciden']} | Speedup: {resumen['speedup']}x")
    print()
    print(cargar_historial_benchmark())

Coinciden: False | Speedup: Nonex

    nº            timestamp archivo_origen  archivo_destino lenguaje_origen  \
0    1  2026-07-14 18:32:34   prueba_pi.py  (Nuevo archivo)          Python   
1    2  2026-07-14 18:44:35   prueba_pi.py    prueba_pi.cpp          Python   
2    3  2026-07-14 19:07:27   prueba_pi.py    prueba_pi.cpp          Python   
3    4  2026-07-14 19:17:37   prueba_pi.py    prueba_pi.cpp          Python   
4    5  2026-07-14 19:48:48    Prueba_1.py  (Nuevo archivo)          Python   
5    6  2026-07-14 20:20:41    Prueba_1.py  (Nuevo archivo)          Python   
6    7  2026-07-14 20:22:31    Prueba_1.py  (Nuevo archivo)          Python   
7    8  2026-07-14 20:26:19    Prueba_1.py  (Nuevo archivo)          Python   
8    9  2026-07-14 20:26:46    Prueba_1.py  (Nuevo archivo)          Python   
9   10  2026-07-14 20:26:51    Prueba_1.py  (Nuevo archivo)          Python   
10  11  2026-07-14 20:34:20    Prueba_1.py  (Nuevo archivo)          Python   
11  12  2026-07-1

In [64]:
# ============================================================
# SECCIÓN A — BLOQUE 7: Interfaz Gradio — Tab 1 (Conversión)
# ============================================================
LANG_TO_GRCODE = {"Python": "python", "C++": "cpp"}

In [65]:
# --- Callbacks de cascada: filtro tipo → fabricante → modelo ---

def actualizar_fabricantes(filtro_tipo):
    if filtro_tipo == "Todos":
        fabricantes = list(MODELOS.keys())
    else:
        fabricantes = [f for f, v in MODELOS.items() if v["tipo"] == filtro_tipo]
    valor = fabricantes[0] if fabricantes else None
    return gr.update(choices=fabricantes, value=valor)

In [66]:
def actualizar_modelos(fabricante):
    modelos = list(MODELOS.get(fabricante, {}).get("modelos", {}).keys())
    valor = modelos[0] if modelos else None
    return gr.update(choices=modelos, value=valor)

In [67]:
def resolver_modelo(fabricante, nombre_mostrado):
    """Convierte el nombre visible del modelo en el identificador/URL real que se envía a la API."""
    return MODELOS.get(fabricante, {}).get("modelos", {}).get(nombre_mostrado, nombre_mostrado)

In [68]:
# --- Callbacks de lenguaje: etiqueta de botón + resaltado de sintaxis + listado de archivos ---

def actualizar_por_lenguaje_origen(lenguaje):
    archivos = refrescar_listado_archivos(lenguaje, CARPETA_ORIGEN) + ["(Manual)"]
    return (
        gr.update(value=f"Ejecutar {lenguaje}"),
        gr.update(language=LANG_TO_GRCODE.get(lenguaje)),
        gr.update(choices=archivos, value="(Manual)"),
    )

In [69]:
def actualizar_por_lenguaje_destino(lenguaje):
    archivos = refrescar_listado_archivos(lenguaje, CARPETA_DESTINO) + ["(Nuevo archivo)"]
    return (
        gr.update(value=f"Ejecutar {lenguaje}"),
        gr.update(language=LANG_TO_GRCODE.get(lenguaje)),
        gr.update(choices=archivos, value="(Nuevo archivo)"),
    )

In [70]:
# --- Cargar archivo seleccionado en la caja de código ---

def cargar_archivo(nombre_archivo, carpeta):
    if nombre_archivo in (None, "(Manual)", "(Nuevo archivo)"):
        return gr.update(value="")
    return gr.update(value=leer_archivo(nombre_archivo, carpeta))


In [71]:
def borrar_caja():
    return gr.update(value=""), "", None

In [72]:
# --- Convertir código ---

def accion_convertir(fabricante, modelo_mostrado, lenguaje_origen, lenguaje_destino, codigo_origen, instrucciones):
    modelo_real = resolver_modelo(fabricante, modelo_mostrado)
    codigo_destino = ""
    for parcial in convertir_codigo(fabricante, modelo_real, lenguaje_origen, lenguaje_destino,
                                     codigo_origen, instrucciones):
        codigo_destino = parcial
    return gr.update(value=codigo_destino), None, None


In [73]:
def accion_ejecutar_origen(lenguaje, codigo):
    if not codigo or not codigo.strip():
        return "(sin código que ejecutar)", None
    resultado = execute_code(lenguaje, codigo, "temp_origen", CARPETA_ORIGEN)
    texto = f"Error:\n{resultado['stderr']}" if resultado["error"] else f"Resultado: {resultado['stdout']}\nTiempo: {resultado['tiempo']}s"
    return texto, resultado

In [74]:
def accion_ejecutar_destino(lenguaje, codigo):
    if not codigo or not codigo.strip():
        return "(sin código que ejecutar)", None
    resultado = execute_code(lenguaje, codigo, "temp_destino", CARPETA_DESTINO)
    texto = f"Error:\n{resultado['stderr']}" if resultado["error"] else f"Resultado: {resultado['stdout']}\nTiempo: {resultado['tiempo']}s"
    return texto, resultado

In [75]:
def comparar_y_registrar(resultado_origen, resultado_destino, fabricante, modelo,
                          lenguaje_origen, lenguaje_destino, archivo_origen, archivo_destino):
    if resultado_origen is None or resultado_destino is None:
        df = cargar_historial_benchmark()
        return gr.update(), df, contar_registros(df)

    coinciden = (resultado_origen["error"] is None and resultado_destino["error"] is None
                 and resultado_origen["stdout"] == resultado_destino["stdout"])
    speedup = None
    if resultado_origen["tiempo"] and resultado_destino["tiempo"]:
        speedup = round(resultado_origen["tiempo"] / resultado_destino["tiempo"], 2)

    append_jsonl({
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "archivo_origen": archivo_origen, "archivo_destino": archivo_destino,
        "lenguaje_origen": lenguaje_origen, "lenguaje_destino": lenguaje_destino,
        "fabricante": fabricante, "modelo": modelo,
        "tipo_modelo": MODELOS.get(fabricante, {}).get("tipo"),
        "tiempo_origen_s": resultado_origen["tiempo"], "tiempo_destino_s": resultado_destino["tiempo"],
        "resultado_origen": resultado_origen["stdout"], "resultado_destino": resultado_destino["stdout"],
        "resultados_coinciden": coinciden, "speedup": speedup,
    })

    speedup_txt = f"{speedup}x" if speedup else "N/D"
    coincide_txt = "Coinciden ✓" if coinciden else "NO coinciden ✗"
    df = cargar_historial_benchmark()
    return f"{speedup_txt} — {coincide_txt}", df, contar_registros(df)

In [76]:
def guardar_archivos(que_guardar, nombre_base, codigo_origen, codigo_destino, lenguaje_origen, lenguaje_destino):
    if not nombre_base or not nombre_base.strip():
        nombre_base = "convertido"
    nombre_base = nombre_base.strip()

    mensajes = []
    archivos_origen_upd = gr.update()
    archivos_destino_upd = gr.update()

    if que_guardar in ("Origen", "Ambos"):
        if not codigo_origen or not codigo_origen.strip():
            mensajes.append("No hay código origen que guardar.")
        else:
            nombre_archivo = f"{nombre_base}{EXTENSIONES[lenguaje_origen]}"
            escribir_archivo(nombre_archivo, codigo_origen, CARPETA_ORIGEN)
            archivos = refrescar_listado_archivos(lenguaje_origen, CARPETA_ORIGEN) + ["(Manual)"]
            archivos_origen_upd = gr.update(choices=archivos, value=nombre_archivo)
            mensajes.append(f"Origen guardado: {nombre_archivo}")

    if que_guardar in ("Destino", "Ambos"):
        if not codigo_destino or not codigo_destino.strip():
            mensajes.append("No hay código destino que guardar.")
        else:
            nombre_archivo = f"{nombre_base}{EXTENSIONES[lenguaje_destino]}"
            escribir_archivo(nombre_archivo, codigo_destino, CARPETA_DESTINO)
            archivos = refrescar_listado_archivos(lenguaje_destino, CARPETA_DESTINO) + ["(Nuevo archivo)"]
            archivos_destino_upd = gr.update(choices=archivos, value=nombre_archivo)
            mensajes.append(f"Destino guardado: {nombre_archivo}")

    return " | ".join(mensajes), archivos_origen_upd, archivos_destino_upd

In [77]:
# --- Borrado de archivo (con confirmación) ---

def abrir_confirmacion():
    return gr.update(visible=True)

def cancelar_confirmacion():
    return gr.update(visible=False)

def confirmar_borrado(cual, archivo_origen_sel, archivo_destino_sel, lenguaje_origen, lenguaje_destino):
    mensaje = "No se seleccionó ningún archivo válido."
    if cual == "Archivo origen" and archivo_origen_sel not in (None, "(Manual)"):
        eliminar_archivo(archivo_origen_sel, CARPETA_ORIGEN)
        mensaje = f"Eliminado: {archivo_origen_sel}"
    elif cual == "Archivo destino" and archivo_destino_sel not in (None, "(Nuevo archivo)"):
        eliminar_archivo(archivo_destino_sel, CARPETA_DESTINO)
        mensaje = f"Eliminado: {archivo_destino_sel}"

    archivos_origen = refrescar_listado_archivos(lenguaje_origen, CARPETA_ORIGEN) + ["(Manual)"]
    archivos_destino = refrescar_listado_archivos(lenguaje_destino, CARPETA_DESTINO) + ["(Nuevo archivo)"]

    return (
        gr.update(visible=False),
        mensaje,
        gr.update(choices=archivos_origen, value="(Manual)"),
        gr.update(choices=archivos_destino, value="(Nuevo archivo)"),
    )

In [78]:
def reset_pantalla():
    return (
        gr.update(value=""), gr.update(value=""),   # code_origen, code_destino
        gr.update(value=""),                          # instrucciones
        "", "", "",                                    # resultados origen/destino/speedup
        None, None,                                     # estado_resultado_origen, estado_resultado_destino
        "", "",                                          # txt_nombre_guardar, mensaje_guardado
    )

In [79]:
# ============================================================
# Ensamblado de la interfaz
# ============================================================

In [80]:
# Bloque A - Ensamblado de la interfaz Tab 1

In [81]:
# Bloque B - Ensamblado de la interfaz Tab 2

In [90]:
with gr.Blocks(title="Gradio") as demo:
    gr.Markdown("# Tool - Prompts y Conversor de código con IA")

    with gr.Tabs():

        # ================================================================
        # TAB 1 — Gestión de Prompts
        # ================================================================
        with gr.Tab("Gestión de Prompts"):
            gr.Markdown("## Gestión de Prompts")

            with gr.Row():
                with gr.Column():
                    gr.Markdown("### Prompt de sistema")
                    txt_prompt_sistema = gr.Textbox(label="Prompt de sistema (activo)", value=prompt_sistema_activo, lines=6)
                    with gr.Row():
                        btn_guardar_sistema = gr.Button("Guardar como activo")
                        btn_restablecer_sistema = gr.Button("Restablecer por defecto")
                        btn_recargar_sistema = gr.Button("Recargar activo")
                    mensaje_sistema = gr.Textbox(show_label=False, interactive=False)

                with gr.Column():
                    gr.Markdown("### Prompt de usuario")
                    txt_prompt_usuario = gr.Textbox(label="Prompt de usuario (activo)", value=prompt_usuario_activo, lines=6)
                    with gr.Row():
                        btn_guardar_usuario = gr.Button("Guardar como activo")
                        btn_restablecer_usuario = gr.Button("Restablecer por defecto")
                        btn_recargar_usuario = gr.Button("Recargar activo")
                    mensaje_usuario = gr.Textbox(show_label=False, interactive=False)

            with gr.Accordion("Enriquecer un prompt con IA", open=False):
                radio_cual_prompt = gr.Radio(["Prompt de sistema", "Prompt de usuario"], value="Prompt de sistema", label="¿Qué prompt quieres mejorar?")
                txt_instrucciones_enriquecer = gr.Textbox(label="Instrucciones para la IA", placeholder="ej. pide que sea más estricto con el formato de salida")

                _fabricantes_enriq_iniciales = list(MODELOS_ENRIQUECIMIENTO.keys())
                _fabricante_enriq_inicial = _fabricantes_enriq_iniciales[0] if _fabricantes_enriq_iniciales else None
                _modelos_enriq_iniciales = list(MODELOS_ENRIQUECIMIENTO.get(_fabricante_enriq_inicial, {}).keys())
                _modelo_enriq_inicial = _modelos_enriq_iniciales[0] if _modelos_enriq_iniciales else None

                with gr.Row():
                    dd_fabricante_enriquecimiento = gr.Dropdown(_fabricantes_enriq_iniciales, value=_fabricante_enriq_inicial, label="Fabricante")
                    dd_modelo_enriquecimiento = gr.Dropdown(_modelos_enriq_iniciales, value=_modelo_enriq_inicial, label="Modelo")
                    btn_actualizar_modelos_ia = gr.Button("Actualizar modelos IA")

                btn_enriquecer = gr.Button("Enriquecer con IA", variant="primary")
                mensaje_enriquecer = gr.Textbox(show_label=False, interactive=False)
                txt_propuesta = gr.Textbox(label="Propuesta generada", lines=6)

                with gr.Row():
                    btn_aceptar_propuesta = gr.Button("Aceptar propuesta")
                    btn_descartar_propuesta = gr.Button("Descartar")

        # ================================================================
        # TAB 2 — Conversión de código
        # ================================================================
        with gr.Tab("Conversión de código"):
            gr.Markdown("## Conversión de código con IA")

            with gr.Row():
                # ---------- COLUMNA 1: Configuración ----------
                with gr.Column(scale=1):
                    with gr.Accordion("Configuración de lenguajes", open=False):
                        with gr.Row():
                            dd_lenguaje_origen = gr.Dropdown(LANGUAGES, value=LENGUAJE_ORIGEN_DEFAULT, label="Lenguaje origen")
                            dd_lenguaje_destino = gr.Dropdown(LANGUAGES, value=LENGUAJE_DESTINO_DEFAULT, label="Lenguaje destino")

                    with gr.Accordion("Gestión de archivos", open=False):
                        with gr.Row():
                            with gr.Column():
                                dd_archivo_origen = gr.Dropdown(archivos_origen_iniciales + ["(Manual)"], value="(Manual)", label="Archivo origen")
                                btn_cargar_origen = gr.Button("Cargar archivo")
                            with gr.Column():
                                dd_archivo_destino = gr.Dropdown(archivos_destino_iniciales + ["(Nuevo archivo)"], value="(Nuevo archivo)", label="Archivo destino")
                                btn_cargar_destino = gr.Button("Cargar archivo")

                        gr.Markdown("**Guardar código actual en archivo**")
                        with gr.Row():
                            radio_guardar = gr.Radio(["Origen", "Destino", "Ambos"], value="Destino", label="Qué guardar")
                            txt_nombre_guardar = gr.Textbox(label="Nombre de archivo (sin extensión)", placeholder="ej. mi_resultado")
                            btn_guardar = gr.Button("Guardar archivo")
                        mensaje_guardado = gr.Textbox(show_label=False, interactive=False)

                        gr.Markdown("**Eliminar archivo**")
                        btn_eliminar = gr.Button("Eliminar archivo")
                        with gr.Group(visible=False) as panel_confirmacion:
                            gr.Markdown("**Vas a eliminar un archivo. Elige cuál:**")
                            radio_borrado = gr.Radio(["Archivo origen", "Archivo destino"], show_label=False)
                            with gr.Row():
                                btn_confirmar_borrado = gr.Button("Confirmar")
                                btn_cancelar_borrado = gr.Button("Cancelar")
                            mensaje_borrado = gr.Textbox(show_label=False, interactive=False)

                    with gr.Accordion("Tipo de modelo", open=False):
                        filtro_tipo = gr.Radio(["Frontera", "Open-source", "Todos"], value="Frontera", show_label=False)
                        _fabricantes_frontera_iniciales = [f for f, v in MODELOS.items() if v["tipo"] == "Frontera"]
                        _fabricante_inicial = _fabricantes_frontera_iniciales[0] if _fabricantes_frontera_iniciales else None
                        _modelos_iniciales = list(MODELOS.get(_fabricante_inicial, {}).get("modelos", {}).keys())
                        _modelo_inicial = _modelos_iniciales[0] if _modelos_iniciales else None
                        dd_fabricante = gr.Dropdown(_fabricantes_frontera_iniciales, value=_fabricante_inicial, label="Fabricante")
                        dd_modelo = gr.Dropdown(_modelos_iniciales, value=_modelo_inicial, label="Modelo")

                    instrucciones = gr.Textbox(label="Prompt adicional (opcional, solo esta ejecución)", lines=2)

                    btn_convertir = gr.Button("Convertir", variant="primary")
                    btn_reset = gr.Button("Reset pantalla")

                    speedup_out = gr.Textbox(label="Comparativa", interactive=False)

                # ---------- COLUMNA 2: Código origen ----------
                with gr.Column(scale=2):
                    code_origen = gr.Code(label="Código origen", language="python", lines=20, value="")
                    with gr.Row():
                        btn_ejecutar_origen = gr.Button("Ejecutar Python")
                        btn_borrar_origen = gr.Button("Borrar caja")
                    resultado_origen_out = gr.Textbox(label="Resultado origen", interactive=False)

                # ---------- COLUMNA 3: Código destino ----------
                with gr.Column(scale=2):
                    code_destino = gr.Code(label="Código destino", language="cpp", lines=20, value="")
                    with gr.Row():
                        btn_ejecutar_destino = gr.Button("Ejecutar C++")
                        btn_borrar_destino = gr.Button("Borrar caja")
                    resultado_destino_out = gr.Textbox(label="Resultado destino", interactive=False)

            estado_resultado_origen = gr.State(None)
            estado_resultado_destino = gr.State(None)

        # ================================================================
        # TAB 3 — Historial de ejecuciones
        # ================================================================
        with gr.Tab("Historial de ejecuciones") as tab_historial:
                _historial_inicial = cargar_historial_benchmark()
                #tabla_historial = gr.Dataframe(label="Registros", value=_historial_inicial)
                tabla_historial = gr.Dataframe(label="Historial de ejecuciones", value=_historial_inicial, wrap=False)
                total_historial_out = gr.Markdown(contar_registros(_historial_inicial))

                with gr.Row():
                    btn_gestionar_historial = gr.Button("Gestionar historial")
    
                with gr.Group(visible=False) as panel_confirmacion_historial:
                    gr.Markdown("**¿Qué quieres hacer con el historial de ejecuciones?**")
                    radio_historial = gr.Radio(["Vaciar todo", "Mantener últimos 10"], show_label=False)
                    with gr.Row():
                        btn_confirmar_historial = gr.Button("Confirmar")
                        btn_cancelar_historial = gr.Button("Cancelar")
                    mensaje_historial = gr.Textbox(show_label=False, interactive=False)

        # ================================================================
        # TAB 4 — Ayuda
        # ================================================================
        with gr.Tab("Ayuda"):
            gr.Markdown(texto_ayuda())
    
   # ================================================================
    # EVENTOS — Tab Gestión de Prompts
    # ================================================================
    btn_guardar_sistema.click(guardar_prompt_sistema, txt_prompt_sistema, mensaje_sistema)
    btn_restablecer_sistema.click(restablecer_prompt_sistema, None, [txt_prompt_sistema, mensaje_sistema])
    btn_recargar_sistema.click(recargar_prompt_sistema, None, [txt_prompt_sistema, mensaje_sistema])

    btn_guardar_usuario.click(guardar_prompt_usuario, txt_prompt_usuario, mensaje_usuario)
    btn_restablecer_usuario.click(restablecer_prompt_usuario, None, [txt_prompt_usuario, mensaje_usuario])
    btn_recargar_usuario.click(recargar_prompt_usuario, None, [txt_prompt_usuario, mensaje_usuario])

    dd_fabricante_enriquecimiento.change(actualizar_modelos_enriquecimiento, dd_fabricante_enriquecimiento, dd_modelo_enriquecimiento)
    btn_actualizar_modelos_ia.click(actualizar_modelos_ia_api, dd_fabricante_enriquecimiento, [dd_modelo_enriquecimiento, mensaje_enriquecer])

    btn_enriquecer.click(
        enriquecer_prompt,
        [radio_cual_prompt, txt_prompt_sistema, txt_prompt_usuario, txt_instrucciones_enriquecer,
         dd_fabricante_enriquecimiento, dd_modelo_enriquecimiento],
        [mensaje_enriquecer, txt_propuesta],
    )

    btn_aceptar_propuesta.click(
        aceptar_propuesta,
        [radio_cual_prompt, txt_propuesta],
        [txt_prompt_sistema, txt_prompt_usuario, mensaje_enriquecer],
    )
    btn_descartar_propuesta.click(descartar_propuesta, None, txt_propuesta)
    
    # ================================================================
    # EVENTOS — Tab Conversión de código
    # ================================================================
    dd_lenguaje_origen.change(actualizar_por_lenguaje_origen, dd_lenguaje_origen,
                              [btn_ejecutar_origen, code_origen, dd_archivo_origen])
    dd_lenguaje_destino.change(actualizar_por_lenguaje_destino, dd_lenguaje_destino,
                               [btn_ejecutar_destino, code_destino, dd_archivo_destino])

    btn_cargar_origen.click(lambda n: cargar_archivo(n, CARPETA_ORIGEN), dd_archivo_origen, code_origen)
    btn_cargar_destino.click(lambda n: cargar_archivo(n, CARPETA_DESTINO), dd_archivo_destino, code_destino)

    btn_borrar_origen.click(borrar_caja, None, [code_origen, resultado_origen_out, estado_resultado_origen])
    btn_borrar_destino.click(borrar_caja, None, [code_destino, resultado_destino_out, estado_resultado_destino])

    filtro_tipo.change(actualizar_fabricantes, filtro_tipo, dd_fabricante)
    dd_fabricante.change(actualizar_modelos, dd_fabricante, dd_modelo)

    btn_convertir.click(
        accion_convertir,
        [dd_fabricante, dd_modelo, dd_lenguaje_origen, dd_lenguaje_destino, code_origen, instrucciones],
        [code_destino, estado_resultado_origen, estado_resultado_destino],
    )

    btn_ejecutar_origen.click(
        accion_ejecutar_origen, [dd_lenguaje_origen, code_origen],
        [resultado_origen_out, estado_resultado_origen]
    ).then(
        comparar_y_registrar,
        [estado_resultado_origen, estado_resultado_destino, dd_fabricante, dd_modelo,
         dd_lenguaje_origen, dd_lenguaje_destino, dd_archivo_origen, dd_archivo_destino],
        [speedup_out, tabla_historial, total_historial_out],
    )

    btn_ejecutar_destino.click(
        accion_ejecutar_destino, [dd_lenguaje_destino, code_destino],
        [resultado_destino_out, estado_resultado_destino]
    ).then(
        comparar_y_registrar,
        [estado_resultado_origen, estado_resultado_destino, dd_fabricante, dd_modelo,
         dd_lenguaje_origen, dd_lenguaje_destino, dd_archivo_origen, dd_archivo_destino],
        [speedup_out, tabla_historial, total_historial_out],
    )

    btn_guardar.click(
        guardar_archivos,
        [radio_guardar, txt_nombre_guardar, code_origen, code_destino, dd_lenguaje_origen, dd_lenguaje_destino],
        [mensaje_guardado, dd_archivo_origen, dd_archivo_destino],
    )

    btn_eliminar.click(abrir_confirmacion, None, panel_confirmacion)
    btn_cancelar_borrado.click(cancelar_confirmacion, None, panel_confirmacion)
    btn_confirmar_borrado.click(
        confirmar_borrado,
        [radio_borrado, dd_archivo_origen, dd_archivo_destino, dd_lenguaje_origen, dd_lenguaje_destino],
        [panel_confirmacion, mensaje_borrado, dd_archivo_origen, dd_archivo_destino],
    )

    btn_reset.click(
        reset_pantalla, None,
        [code_origen, code_destino, instrucciones, resultado_origen_out, resultado_destino_out, speedup_out,
         estado_resultado_origen, estado_resultado_destino, txt_nombre_guardar, mensaje_guardado],
    )
    
    # ================================================================
    # EVENTOS — Tab Historial de ejecuciones
    # ================================================================
    #tab_historial.select(cargar_historial_al_entrar, None, [tabla_historial, total_historial_out])
    btn_gestionar_historial.click(lambda: gr.update(visible=True), None, panel_confirmacion_historial)
    btn_cancelar_historial.click(lambda: gr.update(visible=False), None, panel_confirmacion_historial)
    btn_confirmar_historial.click(
        gestionar_historial, radio_historial, [mensaje_historial, tabla_historial, total_historial_out]
    ).then(lambda: gr.update(visible=False), None, panel_confirmacion_historial)


    # ================================================================
    # PIE DE PÁGINA
    # ================================================================
    gr.Markdown(f"---\n<small>José María Rivas | Práctica LLM Engineering · {datetime.now().strftime('%d/%m/%Y')}</small>")

demo.launch()

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.
